In [1]:
# selector agent
from autogen_agentchat.teams import SelectorGroupChat
# UserProxyAgent 는 사용자의 입력을 받아 에이전트에게 전달하는 역할을 함
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import web_search_tool, save_report_to_md

In [2]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
)

In [3]:
""" 
round-robin 방식으로 email-optimizer-team 을 구현한 방식과 달리 
system_message(prompt)뿐만 아니라 selector agent가 참고할 설명을 넘겨야함
"""
research_planner = AssistantAgent(
    "research_planner",
    description="A strategic research coordinator that breaks down complex questions into research subtasks",
    model_client=model_client,
    system_message="""You are a research planning specialist. Your job is to create a focused research plan.

For each research question, create a FOCUSED research plan with:

1. **Core Topics**: 2-3 main areas to investigate
2. **Search Queries**: Create 3-5 specific search queries covering:
   - Latest developments and news
   - Key statistics or data
   - Expert analysis or studies
   - Future outlook

Keep the plan focused and achievable. Quality over quantity.""",
)

research_agent = AssistantAgent(
    "research_agent",
    description="A web research specialist that searches and extracts content",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""You are a web research specialist. Your job is to conduct focused searches based on the research plan.

RESEARCH STRATEGY:
1. **Execute 3-5 searches** from the research plan
2. **Extract key information** from the results:
   - Main facts and statistics
   - Recent developments
   - Expert opinions
   - Important context

3. **Quality focus**:
   - Prioritize authoritative sources
   - Look for recent information (within 2 years)
   - Note diverse perspectives

After completing the searches from the plan, summarize what you found. Your goal is to gather 5-10 quality sources.""",
)

research_analyst = AssistantAgent(
    "research_analyst",
    description="An expert analyst that creates research reports",
    model_client=model_client,
    system_message="""You are a research analyst. Create a comprehensive report from the gathered research.

CREATE A RESEARCH REPORT with:

## Executive Summary
- Key findings and conclusions
- Main insights

## Background & Current State
- Current landscape
- Recent developments
- Key statistics and data

## Analysis & Insights
- Main trends
- Different perspectives
- Expert opinions

## Future Outlook
- Emerging trends
- Predictions
- Implications

## Sources
- List all sources used

Write a clear, well-structured report based on the research gathered. End with "REPORT_COMPLETE" when finished.""",
)

quality_reviewer = AssistantAgent(
    "quality_reviewer",
    description="A quality assurance specialist that evaluates research completeness and accuracy",
    tools=[save_report_to_md],
    model_client=model_client,
    system_message="""You are a quality reviewer. Your job is to check if the research analyst has produced a complete research report.

Look for:
- A comprehensive research report from the research analyst that ends with "REPORT_COMPLETE"
- The research question is fully answered
- Sources are cited and reliable
- The report includes summary, key information, analysis, and sources

When you see a complete research report that ends with "REPORT_COMPLETE":
1. First, use the save_report_to_md tool to save the report to report.md
2. Then say: "The research is complete. The report has been saved to report.md. Please review the report and let me know if you approve it or need additional research."

If the research analyst has NOT yet created a complete report, tell them to create one now.""",
)

# gap, 조사에 빠진 곳이 있는지만 확인
research_enhancer = AssistantAgent(
    "research_enhancer",
    description="A specialist that identifies critical gaps only",
    model_client=model_client,
    system_message="""You are a research enhancement specialist. Your job is to identify ONLY CRITICAL gaps.

Review the research and ONLY suggest additional searches if there are MAJOR gaps like:
- Completely missing recent developments (last 6 months)
- No statistics or data at all
- Missing a crucial perspective that was specifically asked for

If the research covers the basics reasonably well, say: "The research is sufficient to proceed with the report."

Only suggest 1-2 additional searches if absolutely necessary. We prioritize getting a good report done rather than perfect coverage.""",
)

user_proxy = UserProxyAgent(
    "user_proxy",
    description="Human reviewer who can request additional research or approve final results",
    input_func=input,
)

In [4]:
# 가장 많은 노력을 들여야 하는 구문.
## 모든게 어떻게 작동해야 하는지 하나하나 적어야 함

selector_prompt = """
Choose the best agent for the current task based on the conversation history:

{roles}

Current conversation:
{history}

Available agents:
- research_planner: Plan the research approach (ONLY at the start)
- research_agent: Search for and extract content from web sources (after planning)
- research_enhancer: Identify CRITICAL gaps only (use sparingly)
- research_analyst: Write the final research report
- quality_reviewer: Check if a complete report exists
- user_proxy: Ask the human for feedback

WORKFLOW:
1. If no planning done yet → select research_planner
2. If planning done but no research → select research_agent  
3. After research_agent completes initial searches → select research_enhancer ONCE
4. If enhancer says "sufficient to proceed" → select research_analyst
5. If enhancer suggests critical searches → select research_agent ONCE more then research_analyst
6. If research_analyst said "REPORT_COMPLETE" → select quality_reviewer
7. If quality_reviewer asked for user feedback → select user_proxy

IMPORTANT: After research_agent has searched 2 times maximum, proceed to research_analyst regardless.

Pick the agent that should work next based on this workflow."""

In [5]:
text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(max_messages=50)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst,
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    # allow_repeated_speaker=True,
    termination_condition=termination_condition,
)

In [6]:
await Console(
    # team.run_stream(task="Research about the new development in Nuclear Energy"),
    team.run_stream(task="북중미월드컵 대한민국 대표팀 선수단에 최적화된 전술 분석 및 제안"),
)

---------- TextMessage (user) ----------
북중미월드컵 대한민국 대표팀 선수단에 최적화된 전술 분석 및 제안
---------- TextMessage (research_planner) ----------
### FOCUSED Research Plan: 최적화된 전술 분석 및 제안 for 대한민국 대표팀 (북중미월드컵)

#### 1. Core Topics:
   - **전술적 접근 방식**: 대한민국 대표팀의 기존 전술 분석 및 월드컵에서의 효용.
   - **선수 성능 분석**: 선수별 능력치, 포지션, 체력 및 폼 분석.
   - **경기 상대 분석**: 예선 및 본선에서의 주요 상대 팀의 전술 및 강약점.

#### 2. Search Queries:
   - **최신 개발 및 뉴스**:
     1. "2023 북중미월드컵 대한민국 대표팀 전술 분석"
     2. "대한민국 축구팀 최신 구성 및 전략 변화"
  
   - **핵심 통계 또는 데이터**:
     1. "대한민국 대표팀 선수 통계 2023 월드컵 예선"
     2. "2023 월드컵 대비 대한민국 선수 개인 성능 데이터"
  
   - **전문가 분석 또는 연구**:
     1. "전문가 의견: 대한민국 축구팀의 전술적 강점과 약점"
     2. "축구 전술 연구: 월드컵에서의 효과적인 전략 사례 분석"
  
   - **미래 전망**:
     1. "대한민국 축구팀 2023 월드컵 전망 및 기대"
     2. "앞으로 5년간의 대한민국 축구팀 전략적 발전 방향" 

이 연구 계획은 대한민국 대표팀이 북중미월드컵에서 최적의 성과를 낼 수 있도록 전술을 면밀히 분석하고 제안하는 데 집중됩니다. 각 핵심 주제를 바탕으로 구체적인 검색 쿼리를 통해 신뢰성 있는 데이터와 의견을 수집할 수 있습니다.
---------- ToolCallRequestEvent (research_agent) ----------
[FunctionCall(id='call_6NGsbN

TaskResult(messages=[TextMessage(id='311790d3-fdf0-45e3-bf35-03e54e1973a6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 1, 9, 31, 5, 767939, tzinfo=datetime.timezone.utc), content='북중미월드컵 대한민국 대표팀 선수단에 최적화된 전술 분석 및 제안', type='TextMessage'), TextMessage(id='c5e673c8-6485-4cc9-ab43-a697fff1d7db', source='research_planner', models_usage=RequestUsage(prompt_tokens=135, completion_tokens=402), metadata={}, created_at=datetime.datetime(2026, 4, 1, 9, 31, 15, 645373, tzinfo=datetime.timezone.utc), content='### FOCUSED Research Plan: 최적화된 전술 분석 및 제안 for 대한민국 대표팀 (북중미월드컵)\n\n#### 1. Core Topics:\n   - **전술적 접근 방식**: 대한민국 대표팀의 기존 전술 분석 및 월드컵에서의 효용.\n   - **선수 성능 분석**: 선수별 능력치, 포지션, 체력 및 폼 분석.\n   - **경기 상대 분석**: 예선 및 본선에서의 주요 상대 팀의 전술 및 강약점.\n\n#### 2. Search Queries:\n   - **최신 개발 및 뉴스**:\n     1. "2023 북중미월드컵 대한민국 대표팀 전술 분석"\n     2. "대한민국 축구팀 최신 구성 및 전략 변화"\n  \n   - **핵심 통계 또는 데이터**:\n     1. "대한민국 대표팀 선수 통계 2023 월드컵 예선"\n     2. "2023 월드컵 대비 대한민국 선수 